# TER — 텍스트 기반 감정 분류 (AI Hub 5차년도·2차 CSV)

`stt_03_train_ser.ipynb`와 **동일한 7감정 라벨·정규화·train/val/test 비율(70/15/15)** 을 사용합니다.  
나중에 **SER logits + TER logits** 융합 시 `LABEL2ID` / `id2label` 순서가 맞습니다.

- 입력: `발화문` 컬럼
- 정답: `상황` → SER와 같은 `normalize_situation` 규칙
- 베이스: **Kakao Kanana Nano Instruct** (`kakaocorp/kanana-nano-2.1b-instruct`, Llama 계열 → `LlamaForSequenceClassification`)

In [ ]:
import os
import sys

# SER 노트북과 같이 Colab Drive를 쓰는 경우 주석 해제
# from google.colab import drive
# drive.mount("/content/drive")

PROJECT_ROOT = os.environ.get("KYUL_PROJECT_ROOT", "").strip()
if not PROJECT_ROOT:
    _colab = "/content/drive/MyDrive/Colab Notebooks/kyul_stt"
    if os.path.isdir(_colab):
        PROJECT_ROOT = _colab
    else:
        PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

SRC = os.path.join(PROJECT_ROOT, "src")
if os.path.isdir(SRC) and SRC not in sys.path:
    sys.path.insert(0, SRC)

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", os.getcwd())

In [ ]:
import os

ENCODING = "cp949"
TEXT_COL = "발화문"
SITUATION_COL = "상황"

_csv_candidates = [
    os.path.join(PROJECT_ROOT, "data", "5_csv", "5차년도_2차.csv"),
    os.path.join(PROJECT_ROOT, "train_data_with_pandas", "5차년도_2차.csv"),
]
CSV_PATH = next((p for p in _csv_candidates if os.path.isfile(p)), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "5차년도_2차.csv 를 찾을 수 없습니다. "
        f"다음 중 하나에 두세요: {_csv_candidates} 또는 KYUL_PROJECT_ROOT 설정"
    )
print("CSV_PATH:", CSV_PATH)

In [ ]:
import os

OUT_DIR = os.path.join(PROJECT_ROOT, "runs", "ter_aihub_nb")
# Kakao Kanana (Llama 아키텍처) — HF: kakaocorp/kanana-nano-2.1b-instruct
BASE_MODEL = "kakaocorp/kanana-nano-2.1b-instruct"

In [ ]:
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LR = 1e-5
SEED = 42
WEIGHT_DECAY = 0.01

In [ ]:
from __future__ import annotations

import os
import random
from typing import Any

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

os.environ.setdefault("TRANSFORMERS_NO_TF", "1")

# SER 노트북과 동일한 클래스 순서 (id2label 인덱스 = SER head 출력 차원과 대응)
LABELS_EN_7 = [
    "happiness",
    "angry",
    "disgust",
    "fear",
    "neutral",
    "sadness",
    "surprise",
]

LABEL2ID = {
    "happiness": 0,
    "angry": 1,
    "disgust": 2,
    "fear": 3,
    "neutral": 4,
    "sadness": 5,
    "surprise": 6,
}

ID2LABEL = {i: lab for lab, i in LABEL2ID.items()}


def normalize_situation(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    t = str(raw).strip().lower()
    if not t:
        return None
    syn = {
        "anger": "angry",
        "angry": "angry",
        "happiness": "happiness",
        "happy": "happiness",
        "disgust": "disgust",
        "fear": "fear",
        "neutral": "neutral",
        "sadness": "sadness",
        "sad": "sadness",
        "surprise": "surprise",
    }
    return syn.get(t)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_text_rows(csv_path: str, encoding: str) -> list[dict[str, Any]]:
    df = pd.read_csv(csv_path, encoding=encoding)
    rows = []
    skip_label, skip_text = 0, 0
    for _, r in df.iterrows():
        lab = normalize_situation(r.get(SITUATION_COL))
        if lab is None or lab not in LABEL2ID:
            skip_label += 1
            continue
        text = r.get(TEXT_COL)
        if text is None or (isinstance(text, float) and np.isnan(text)):
            skip_text += 1
            continue
        text = str(text).strip()
        if not text:
            skip_text += 1
            continue
        rows.append(
            {
                "text": text,
                "label": LABEL2ID[lab],
                "label_en": lab,
                "wav_id": str(r["wav_id"]).strip() if "wav_id" in r else "",
            }
        )
    print(f"유효 {len(rows)} / 전체 {len(df)} (라벨스킵 {skip_label}, 빈텍스트 {skip_text})")
    return rows


def split_stratified(
    rows, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42
):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6
    if len(rows) < 3:
        return rows, [], []
    y = [int(r["label"]) for r in rows]
    try:
        train_rows, tmp = train_test_split(
            rows, test_size=(1.0 - train_ratio), random_state=seed, stratify=y
        )
        vf = val_ratio / (val_ratio + test_ratio)
        y_tmp = [int(r["label"]) for r in tmp]
        val_rows, test_rows = train_test_split(
            tmp, test_size=(1.0 - vf), random_state=seed, stratify=y_tmp
        )
    except ValueError:
        rng = random.Random(seed)
        tmp = rows[:]
        rng.shuffle(tmp)
        n = len(tmp)
        n_train = max(1, int(n * train_ratio))
        n_val = max(1, int(n * val_ratio))
        train_rows = tmp[:n_train]
        val_rows = tmp[n_train : n_train + n_val]
        test_rows = tmp[n_train + n_val :]
        if not test_rows:
            test_rows = val_rows
    return train_rows, val_rows, test_rows


set_seed(SEED)
rows = load_text_rows(CSV_PATH, ENCODING)
if not rows:
    raise RuntimeError("유효 샘플 0 — CSV_PATH·컬럼명 확인")

train_rows, val_rows, test_rows = split_stratified(rows, seed=SEED)
print("train", len(train_rows), "val", len(val_rows), "test", len(test_rows))
print("예시:", train_rows[0])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if getattr(tokenizer, "pad_token", None) is None:
    tokenizer.pad_token = tokenizer.eos_token
# Llama 계열 배치 분류: 오른쪽 패딩이면 마지막 토큰이 pad가 되기 쉬움 → 왼쪽 패딩
tokenizer.padding_side = "left"


def rows_to_hf_dataset(split_rows):
    return Dataset.from_dict(
        {
            "text": [r["text"] for r in split_rows],
            "labels": [r["label"] for r in split_rows],
        }
    )


train_ds = rows_to_hf_dataset(train_rows)
val_ds = rows_to_hf_dataset(val_rows)
test_ds = rows_to_hf_dataset(test_rows)


def tokenize_batch(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

In [ ]:
_use_bf16 = bool(
    torch.cuda.is_available()
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)
_use_fp16 = bool(torch.cuda.is_available() and not _use_bf16)
torch_dtype = torch.bfloat16 if _use_bf16 else torch.float32

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS_EN_7),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
    torch_dtype=torch_dtype,
)
model.config.pad_token_id = tokenizer.pad_token_id


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }


collator = DataCollatorWithPadding(tokenizer)

out_sub = os.path.join(OUT_DIR, BASE_MODEL.replace("/", "_"))
os.makedirs(out_sub, exist_ok=True)

targs = TrainingArguments(
    output_dir=out_sub,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    bf16=_use_bf16,
    fp16=_use_fp16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=targs,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

def epoch_metrics_table(trainer):
    import pandas as pd

    logs = trainer.state.log_history
    eval_logs = [x for x in logs if "eval_loss" in x]
    train_logs = [x for x in logs if "loss" in x and "eval_loss" not in x]
    rows = []
    prev_e = 0.0
    for ev in sorted(eval_logs, key=lambda x: float(x["epoch"])):
        e = float(ev["epoch"])
        chunk = [t["loss"] for t in train_logs if prev_e <= float(t.get("epoch", 0)) < e]
        train_loss = sum(chunk) / len(chunk) if chunk else float("nan")
        rows.append(
            {
                "Epoch": int(round(e)),
                "Training Loss": train_loss,
                "Validation Loss": ev["eval_loss"],
                "Accuracy": ev.get("eval_accuracy", float("nan")),
                "Macro F1": ev.get("eval_macro_f1", float("nan")),
            }
        )
        prev_e = e
    return pd.DataFrame(rows)


summary_df = epoch_metrics_table(trainer)
try:
    from IPython.display import display

    display(summary_df)
except Exception:
    print(summary_df.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

print("val:", trainer.evaluate(eval_dataset=val_ds))
print("test:", trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test"))

MODEL_DIR = os.path.join(out_sub, "model")
os.makedirs(MODEL_DIR, exist_ok=True)
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print("저장 완료:", MODEL_DIR)
print("SER와 합칠 때 logits 차원 순서:", LABELS_EN_7)